In [14]:
#Import the necessary libraries and load environment variables
import os

from anthropic import Anthropic
from dotenv import load_dotenv
from policy import POLICY
import sqlite3
import requests

load_dotenv()

client = Anthropic()


python-dotenv could not parse statement starting at line 4
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 8
python-dotenv could not parse statement starting at line 9
python-dotenv could not parse statement starting at line 10


In [ ]:
import json
from operator import mod
import uuid
import sqlite3
from datetime import datetime, timezone
from typing import Dict, Tuple

'''
Make changes as per retry_policy_v1.json structure. 
The current implementation assumes a specific structure for the retry policy JSON file. 
If the structure of retry_policy_v1.json is different, you may need to adjust the code accordingly. 
For example, if the error classifications are nested differently or if there are additional fields, 
you will need to modify the classify_error and should_retry methods to correctly parse and utilize 
the information from the JSON file.
'''
class ClassifierMapping:
    def __init__(self, mapping_file="classifier_mapping.json"):
        with open(mapping_file) as f:
            self.mapping = json.load(f)
            self.by_exception = self.mapping.get("by_exception", {})
            self.by_status = self.mapping.get("by_status", {})
            self.status_ranges = self.mapping.get("status_ranges", [])

    def _names(self, cls) -> list[str]:
        """Return list of names to check for a class."""
        mod, qn = cls.__module__, cls.__qualname__
        root = mod.split(".")[0]
        return [f"{mod}.{qn}", f"{root}.{qn}", qn]   # full → root package → bare
    
    def get_error_category(self, exc: Exception | None=None, response=None) -> tuple[str, bool]:
        """Returns (category, was_default). Exception path wins over status path."""
        if exc is not None:
            for cls in type(exc).__mro__:
                for name in self._names(cls):
                    if name in self.by_exception:
                        return self.by_exception[name], False
            return "unknown", True

        if response is not None and response.status_code >= 400:
            code = str(response.status_code)
            if code in self.by_status:
                return self.by_status[code], False
            for r in self.status_ranges:
                if r["min"] <= response.status_code <= r["max"]:
                    return r["category"], False
            return "unknown", True

        return "unknown", True

class RetryPolicy:
    def __init__(self, policy_file="retry_policy_v1.json"):
        with open(policy_file) as f:
            self.config = json.load(f)
        self.policy_version = self.config["policy_version"]
    
    def classify_error(self, error_category: str) -> Dict:
        """Classify error, fallback to default if unknown."""
        if error_category in self.config["error_classifications"]:
            return self.config["error_classifications"][error_category], False
        return self.config["default_classification"], True
    
    def should_retry(self, error_category: str, current_attempt: int) -> Tuple[bool, str]:
        """Orchestrator asks: should I retry?"""
        classification, is_default = self.classify_error(error_category)
        
        if not classification.get("is_retryable", False):
            return False, f"{error_category} is non-retryable"
        
        max_attempts = classification.get("max_retries", 0)
        if current_attempt >= max_attempts:
            return False, f"Max attempts ({max_attempts}) exhausted"
        
        return True, classification.get("description", "Retrying per policy")

class AuditLogger:
    def __init__(self, db_path="orchestrator.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("PRAGMA journal_mode=WAL;")
        self.create_tables()
    
    def create_tables(self):
        self.conn.executescript("""
            CREATE TABLE IF NOT EXISTS events (
                event_id                        TEXT PRIMARY KEY,
                correlation_id                  TEXT NOT NULL,
                created_at                      TEXT NOT NULL,
                event_type                      TEXT NOT NULL,

                tool_name                       TEXT,
                tool_input                      TEXT,
                tool_output                     TEXT,
                tool_use_id                     TEXT,
                user_query_text                 TEXT,

                error_category                  TEXT,
                error_type                      TEXT,
                error_message                   TEXT,
                tool_reported_retryable         BOOLEAN,
                policy_handling                 TEXT,
                classification_conflict         BOOLEAN,
                was_classified_as_default       BOOLEAN,

                error_source_event_id           TEXT,
                retry_decision_source_event_id  TEXT,
                decision                        TEXT,
                reason                          TEXT,
                retry_budget_remaining          INTEGER,
                retry_policy_version            TEXT,
                attempt_number                  INTEGER,

                final_answer_context            TEXT,
                context_source_event_id         TEXT,
                final_answer_text               TEXT,

                model_version                   TEXT,
                user_id                         TEXT,

                FOREIGN KEY(error_source_event_id)
                    REFERENCES events(event_id),
                FOREIGN KEY(retry_decision_source_event_id)
                    REFERENCES events(event_id),
                FOREIGN KEY(context_source_event_id)
                    REFERENCES events(event_id)
            );

            CREATE UNIQUE INDEX IF NOT EXISTS idx_unique_correlation_tool
            ON events(correlation_id, event_type, tool_use_id)
            WHERE tool_use_id IS NOT NULL;

            CREATE INDEX IF NOT EXISTS idx_correlation_time
            ON events(correlation_id, created_at);
        """)
        self.conn.commit()
    
    def log_event(self, event_data: Dict):
        #import inspect
        #frame = inspect.currentframe().f_back
        #print(f"log_event called from: {frame.f_code.co_filename}:{frame.f_lineno}")
        """Log event. Idempotent on event_id."""
        #print("INSERTING", event_data)
        try:
            self.conn.execute("""
            INSERT INTO events (
                event_id, correlation_id, created_at, event_type,
                tool_name, tool_input, tool_output, tool_use_id, user_query_text,
                error_category, error_type, error_message,
                tool_reported_retryable, policy_handling, classification_conflict,
                was_classified_as_default,
                error_source_event_id, retry_decision_source_event_id,
                decision, reason, retry_budget_remaining,
                retry_policy_version, attempt_number,
                final_answer_context, context_source_event_id, final_answer_text,
                model_version, user_id
            ) VALUES (
                :event_id, :correlation_id, :created_at, :event_type,
                :tool_name, :tool_input, :tool_output, :tool_use_id, :user_query_text,
                :error_category, :error_type, :error_message,
                :tool_reported_retryable, :policy_handling, :classification_conflict,
                :was_classified_as_default,
                :error_source_event_id, :retry_decision_source_event_id,
                :decision, :reason, :retry_budget_remaining,
                :retry_policy_version, :attempt_number,
                :final_answer_context, :context_source_event_id, :final_answer_text,
                :model_version, :user_id
            )
        """, event_data)
            self.conn.commit()
        except sqlite3.IntegrityError as e:
            if "UNIQUE" in str(e):
                print(f"[audit] duplicate skipped: {event_data.get('event_id')} | type: {event_data.get('event_type')}")
                import traceback
                #traceback.print_stack()    # shows exactly where log_event was called from
            else:
                raise
        except Exception as e:
            print(f"[audit] INSERT FAILED: {type(e).__name__}: {e}")
            raise
    
    def create_event(self, event_type: str, **kwargs) -> Dict:
        """Factory for creating events."""
        base = {
            "event_id": str(uuid.uuid4()),
            "created_at": datetime.now(timezone.utc).isoformat(),
            "event_type": event_type,
            "correlation_id": None,
            "tool_name": None,
            "tool_input": None,
            "tool_output": None,
            "tool_use_id": None,
            "user_query_text": None,
            "error_category": None,
            "error_type": None,
            "error_message": None,
            "tool_reported_retryable": None,
            "policy_handling": None,
            "classification_conflict": None,
            "was_classified_as_default": None,
            "error_source_event_id": None,
            "retry_decision_source_event_id": None,
            "decision": None,
            "reason": None,
            "retry_budget_remaining": None,
            "retry_policy_version": None,
            "attempt_number": None,
            "final_answer_context": None,
            "context_source_event_id": None,
            "final_answer_text": None,
            "model_version": None,
            "user_id": None,
        }
        base.update(kwargs)
        return base
    
    def get_timeline(self, correlation_id: str):
        """Retrieve full timeline for a conversation."""
        cursor = self.conn.execute("""
        SELECT * 
        FROM events
        WHERE correlation_id = ?
        ORDER BY created_at
        """, (correlation_id,))
        return cursor.fetchall()

    def close(self):
        if self.conn:
            self.conn.close()
            self.conn = None

In [16]:
#Create a function to add user messages to the conversation
#Create a function to add assistant messages to the conversation
#Create a function to converse
from typing import Optional


def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})

def add_assistant_message(messages, content):
    messages.append({"role": "assistant", "content": content})

def converse(messages, model = "claude-haiku-4-5-20251001", max_tokens_to_sample=300,
             temperature: Optional[float] = None,
             stop_sequences: Optional[list[str]] = None,
             tools: Optional[list[dict]] = None,
             system_prompt: Optional[str] = None,
             tool_choice: dict = None) -> dict:

    arg_list = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens_to_sample
    }

    if temperature is not None:
        arg_list["temperature"] = temperature

    if stop_sequences is not None:
        arg_list["stop_sequences"] = stop_sequences

    if tools is not None:
        arg_list["tools"] = tools

    if system_prompt is not None:
        arg_list["system_prompt"] = system_prompt

    response = client.messages.create(
        **arg_list
    )

    return response



In [17]:
# Tool and Schema Definitions

import requests

def get_coordinates_for_city(city: str, timeout_s: float = 10.0) -> dict:
    """
    Returns {"latitude": float, "longitude": float, "city": str}
    or {"error": str, "isRetryable": bool, "attemptedInput": str}
    """
    if not city:
        return {
            "error": "city name is required",
            "isRetryable": False,
            "attemptedInput": city
        }

    url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1"
    
    try:
        response = requests.get(url, timeout=timeout_s)
        response.raise_for_status()
        data = response.json()

        results = data.get("results")
        if not results:
            return {
                "error": f"No location found for '{city}'",
                "isRetryable": False,          # deterministic — retrying won't help
                "attemptedInput": city
            }

        location = results[0]
        lat = location.get("latitude")
        lon = location.get("longitude")

        if lat is None or lon is None:
            return {
                "error": "Location found but coordinates missing",
                "isRetryable": False,
                "attemptedInput": city
            }

        return {
            "latitude": lat,
            "longitude": lon,
            "city": location.get("name", city),
            "country": location.get("country", "")
        }

    except requests.Timeout:
        return {
            "error": "Geocoding service timed out",
            "isRetryable": True,               # transient — worth retrying
            "attemptedInput": city
        }
    except requests.RequestException as e:
        return {
            "error": str(e),
            "isRetryable": True,
            "attemptedInput": city
        }


def get_weather(latitude: float, longitude: float, timeout_s: float = 10.0) -> dict:
    """
    Returns current weather dict
    or {"error": str, "isRetryable": bool, "attemptedInput": dict}
    """
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={latitude}&longitude={longitude}&current_weather=true"
    )

    try:
        response = requests.get(url, timeout=timeout_s)
        response.raise_for_status()
        data = response.json()

        current = data.get("current_weather")
        if not current:
            return {
                "error": "Weather data missing from response",
                "isRetryable": True,
                "attemptedInput": {"latitude": latitude, "longitude": longitude}
            }

        return {
            "temperature_c": current.get("temperature"),
            "windspeed_kmh": current.get("windspeed"),
            "weathercode": current.get("weathercode"),
            "latitude": latitude,
            "longitude": longitude
        }

    except requests.Timeout:
        return {
            "error": "Weather service timed out",
            "isRetryable": True,
            "attemptedInput": {"latitude": latitude, "longitude": longitude}
        }
    except requests.RequestException as e:
        return {
            "error": str(e),
            "isRetryable": True,
            "attemptedInput": {"latitude": latitude, "longitude": longitude}
        }
def get_customer(customer_id: str):
    """
    Get a customer by ID.
    """
    return {
        "id": customer_id,
        "name": "John Doe",
        "email": "john.doe@example.com",
        "acc_type": "business"
    }

def get_business_details(customer_id: str):
    """
    Get business details for a customer.
    """
    return {
        "customer_id": customer_id,
        "business_name": "Doe Enterprises",
        "industry": "Retail",
        "location": "New York, NY"
    }

get_coordinates_for_city_schema = {
    "name": "get_coordinates_for_city",
    "description": "Get coordinates - latitude and longitude - for a city.",
    "input_schema": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "City name"
            }
        },
        "required": []
    }
}    

get_weather_schema = {
    "name": "get_weather",
    "description": "Get the current weather for a given latitude and longitude.",
    "input_schema": {
        "type": "object",
        "properties": {
            "latitude": {
                "type": "number",
                "description": "Latitude of the location"
            },
            "longitude": {
                "type": "number",
                "description": "Longitude of the location"
            }
        },
        "required": []
    }
}

get_customer_schema = {
    "name": "get_customer",
    "description": "Get customer object based on customer ID. It returns property of a customer in json format.",
    "input_schema": {
        "type": "object",
        "properties": {
            "customer_id": {
                "type": "string",
                "description": "Customer ID to lookup"
            }
        },
        "required": []
    }
}

get_business_details_schema = {
    "name": "get_business_details",
    "description": "Get customer's business details based on customer ID. It returns property of a customer's business in json format. Automatically call this tool whenever a customer's account_type is 'business', without asking the user first.",
    "input_schema": {
        "type": "object",
        "properties": {
            "customer_id": {
                "type": "string",
                "description": "Customer ID's business to lookup"
            }
        },
        "required": []
    }
}

In [18]:
def execute_tool(tool_name: str, tool_inputs: dict, timeout_s: float = 10.0) -> dict:
    ADVISORY_RETRYABLE = {"timeout", "network", "upstream_5xx", "rate_limit",
                          "upstream_contract_violation"}

    if tool_name not in ("get_weather", "get_coordinates_for_city"):
        return {
            "ok": False,
            "errorCategory": "internal_error",
            "message": f"Unknown tool: {tool_name}",
            "attemptedInput": tool_inputs,
            "tool_name": tool_name
        }

    # input validation
    try:
        if tool_name == "get_weather":
            lat = float(tool_inputs["latitude"])
            lon = float(tool_inputs["longitude"])
        else:
            city = str(tool_inputs["city"])
    except (KeyError, TypeError, ValueError) as e:
        return {
            "ok": False,
            "errorCategory": "internal_error",
            "message": f"Tool input contract violation: {e}",
            "attemptedInput": tool_inputs,
            "tool_name": tool_name
        }

    # execution — both functions return dicts, symmetric handling
    try:
        if tool_name == "get_weather":
            result = get_weather(lat, lon, timeout_s=timeout_s)
        else:
            result = get_coordinates_for_city(city, timeout_s=timeout_s)
    except Exception as e:
        # safety net — both functions catch their own exceptions
        # this only fires for unexpected bugs (import error, etc.)
        error_classifier = ClassifierMapping()
        category, was_default = error_classifier.get_error_category(exc=e)
        return {
            "ok": False,
            "errorCategory": category,
            "wasClassifiedAsDefault": was_default,
            "toolReportedRetryable": category in ADVISORY_RETRYABLE,
            "attemptedInput": tool_inputs,
            "message": f"{type(e).__name__}: {e}",
            "exceptionType": type(e).__name__,
            "tool_name": tool_name
        }

    # both tools signal failure the same way
    if result.get("error"):
        return {
            "ok": False,
            "errorCategory": "upstream_error",
            "toolReportedRetryable": result.get("isRetryable", False),
            "attemptedInput": tool_inputs,
            "message": result["error"],
            "tool_name": tool_name
        }

    return {"ok": True, "data": result, "tool_name": tool_name}


# This method needs to return a list of results, one per tool actually requested, each tagged with its own tool_use_id
def execute_tools(
    correlation_id: str,
    parsed_response: dict,
    audit_logger: AuditLogger | None,
    original_prompt: str = ""
) -> list:

    if not parsed_response or not parsed_response.get("tools"):
        return []

    results = []

    for tool in parsed_response["tools"]:
        tool_name = tool["name"]
        tool_inputs = tool["inputs"]
        tool_id = tool["tool_id"]

        approved, reason = validate_prerequisites(
            tool_name, tool_inputs, correlation_id, original_prompt
        )

        if approved:
            if audit_logger:
                audit_logger.log_event(audit_logger.create_event(
                    "tool_call",
                    correlation_id=correlation_id,
                    tool_name=tool_name,
                    tool_use_id=tool_id,
                    tool_input=json.dumps(tool_inputs),
                    model_version="claude-haiku-4-5"
                ))

            result = execute_tool(tool_name, tool_inputs)

        else:
            if audit_logger:
                audit_logger.log_event(audit_logger.create_event(
                    "policy_rejection",
                    correlation_id=correlation_id,
                    tool_name=tool_name,
                    tool_use_id=tool_id,
                    reason=reason,
                    model_version="claude-haiku-4-5"
                ))

            result = {
                "ok": False,
                "errorCategory": "policy_rejection",
                "message": reason,
                "tool_name": tool_name    # needed downstream for tool_result assembly
            }

        # carried on both paths — needed by tool_result audit event upstream
        result["tool_input"] = tool_inputs   # keep as dict, not json string
                                             # caller does json.dumps when needed

        results.append({
            "tool_use_id": tool_id,
            "tool_name": tool_name,         # add this — currently missing from the append
            "result": result
        })

    return results

In [19]:
#Write a method to parse assistance message
from anthropic.types import Message, ToolUseBlock, TextBlock


def parse_response(response) -> dict:
    json_response = {"text": "", "tools": []}
    
    if response and isinstance(response, Message):
        
        contents = response.content
        for content in contents:
            if isinstance(content, ToolUseBlock):
                json_response["tools"].append({
                    "inputs": content.input,
                    "name": content.name,
                    "tool_id": content.id
                })
            elif isinstance(content, str):
                json_response["text"] += content
            else:
                json_response["text"] += str(content)
    return json_response

In [20]:
import random

def parse_retry_after(raw) -> int | None:
    if raw is None:
        return None
    try:
        return int(float(raw)) * 1000
    except (TypeError, ValueError):
        return None          # HTTP-date form; ignore for this exercise

def compute_backoff(backoff: dict, attempt: int, retry_after_raw=None) -> tuple[int, bool]:
    """Sleep duration in ms for a 1-based retry attempt.
    Returns (sleep_ms, must_escalate)."""
    strategy = backoff["strategy"]
    base_ms  = backoff["base_ms"]
    max_ms   = backoff.get("max_ms", base_ms)

    if strategy == "fixed":
        raw_ms = base_ms
    elif strategy == "exponential":
        raw_ms = base_ms * (backoff.get("multiplier", 2) ** (attempt - 1))
    else:
        raise ValueError(f"Unknown backoff strategy: {strategy!r}")

    capped = min(raw_ms, max_ms)

    if backoff.get("jitter") == "full":
        sleep_ms = random.uniform(0, capped)
    else:
        sleep_ms = capped

    if backoff.get("honor_retry_after"):
        ra_ms = parse_retry_after(retry_after_raw)
        if ra_ms is not None:
            if ra_ms > max_ms and backoff.get("on_retry_after_exceeds_max") == "escalate":
                return 0, True
            sleep_ms = max(sleep_ms, ra_ms) + random.uniform(0, 1000)  # jitter upward only

    return int(sleep_ms), False

In [21]:
class RetryBudget:
    def __init__(self, cap: int):
        self.cap = cap
        self.used = 0

    def exhausted(self) -> bool:
        return self.used >= self.cap

    def consume(self) -> None:
        if self.exhausted():
            raise RuntimeError(f"Retry budget already exhausted ({self.used}/{self.cap})")
        self.used += 1

    def remaining(self) -> int:
        return self.cap - self.used

In [22]:
b = RetryBudget(4)
assert not b.exhausted()
b.consume(); b.consume()
assert b.remaining() == 2
b.consume(); b.consume()
assert b.exhausted()

try:
    b.consume()
    assert False, "should have raised"
except RuntimeError:
    pass

print("RetryBudget: all assertions passed")

RetryBudget: all assertions passed


In [23]:
def parse_final_response(response: Message) -> str:
    if isinstance(response, Message):
        contents = response.content
        for c in contents:
            if isinstance(c, TextBlock):
                return c.text
        return None
    if isinstance(response, str):
        return response
    return None

In [36]:
def query_history(correlation_id: str, requires: str) -> list[tuple[dict, dict]]:
    """
    Returns (input, output) pairs for successful tool_call events only.
    Filters out error events — a failed call doesn't satisfy a prerequisite.
    """
    conn = sqlite3.connect("orchestrator.db")
    cursor = conn.cursor()
    cursor.execute(
        """
        SELECT tool_input, tool_output FROM events
        WHERE correlation_id = ?
          AND tool_name = ?
          AND event_type = 'tool_result'
          AND tool_output IS NOT NULL
          AND json_extract(tool_output, '$.error') IS NULL
        ORDER BY created_at ASC
        """,
        (str(correlation_id), requires)
    )
    
    results = cursor.fetchall()
    print("RAW ROWS:", results)
    conn.close()
    return [(json.loads(inp), json.loads(out)) for inp, out in results]


def persist_policy_rejection(
    correlation_id: str,
    tool_name: str,
    tool_use_id: str,
    reason: str,
    model_version: str = "claude-haiku-4-5"
):
    conn = sqlite3.connect("orchestrator.db")
    conn.execute(
        """
        INSERT INTO events (
            event_id, correlation_id, created_at, event_type,
            tool_name, tool_use_id, reason, model_version
        ) VALUES (?, ?, datetime('now'), 'policy_rejection', ?, ?, ?, ?)
        """,
        (
            str(uuid.uuid4()),
            str(correlation_id),
            tool_name,
            tool_use_id,
            reason,
            model_version
        )
    )
    conn.commit()
    conn.close()

def validate_prerequisites(
    tool_name: str,
    tool_input: dict,
    correlation_id: str,            # replaces call_history list
    original_prompt: str = ""
) -> tuple[bool, str]:

    prereqs = POLICY.get(tool_name, [])
    print(f"PREREQS: {prereqs}")
    for prereq in prereqs:

        # query DB directly — single source of truth
        prior_calls = query_history(correlation_id, prereq["requires"])
        # returns list of (input_dict, output_dict) tuples
        print(f"PRIOR CALLS : {prior_calls}")
        matched = any(
            all(
                out.get(out_field) == tool_input.get(in_field)
                for out_field, in_field in prereq["on"]
            )
            for inp, out in prior_calls
        )

        if matched:
            if prereq.get("condition"):
                matched_out = next(
                    out for inp, out in prior_calls
                    if all(
                        out.get(out_field) == tool_input.get(in_field)
                        for out_field, in_field in prereq["on"]
                    )
                )
                if not prereq["condition"](matched_out):
                    return False, (
                        f"Prerequisite condition not met for "
                        f"{prereq['requires']}"
                    )
            continue

        if prereq.get("allow_user_supplied") and original_prompt:
            if _values_present_in_prompt(
                tool_input, prereq.get("match_fields", []), original_prompt
            ):
                continue

        return False, (
            f"'{tool_name}' requires '{prereq['requires']}' to have run first, "
            f"or values {prereq.get('match_fields', [])} to be present in the request"
        )

    return True, "approved"


def _values_present_in_prompt(
    tool_input: dict,
    fields: list,
    prompt: str
) -> bool:
    """
    Check that the values the model extracted for `fields`
    actually appear in the original prompt — confirms user-supplied,
    not hallucinated.
    """
    return all(
        str(tool_input.get(f, ""))[:6] in prompt
        for f in fields
        if tool_input.get(f) is not None
    )

In [37]:

import time



def initialize_orchestrator():
    global audit, retry_policy, error_classifier, budget
    if 'audit' in globals() and audit is not None:
        audit.close()
    # In your Exercise 4 orchestrator:
    audit = AuditLogger()
    retry_policy = RetryPolicy()
    budget = RetryBudget(retry_policy.config["global_limits"]["max_total_retries_per_correlation"])

def retry_tool_execution(audit_event, result, classification, is_default) -> dict:
    retry_attempt_event = audit.create_event(
                            "retry_attempt",
                            correlation_id=correlation_id,
                            error_source_event_id=audit_event["event_id"],
                            error_category=result["result"].get("errorCategory"),
                            decision="retry",
                            reason=result["result"]["errorCategory"] + ": attempt " + str(current_attempt) + " of " + str(classification["max_retries"]) + ", budget " + str(classification["max_retries"] - current_attempt) + " remaining",
                            was_classified_as_default=is_default,
                            retry_policy_version=retry_policy.policy_version,
                            retry_budget_remaining=classification["max_retries"] - current_attempt,
                            model_version="claude-haiku-4-5"
                        )
    audit.log_event(retry_attempt_event)
    return execute_tool(result["result"].get("tool_name"), result["result"].get("attemptedInput"))

initialize_orchestrator()
messages  = []
user_query = "get weather for Tokyo"
#user_query = "Tell me a one line story theme"
add_user_message(messages, f"{user_query}")
correlation_id = str(uuid.uuid4())
retry_count_by_category = {}

# Log user query
# The event currently does not include the actual user query text in the event. If you want to log the user query text, you can add it to the event data.
user_query_event = audit.create_event(
    "user_query",
    correlation_id=correlation_id,
    user_query_text=user_query,
    model_version="claude-haiku-4-5"
)
audit.log_event(user_query_event)
tool_choice=(
        {"type": "tool", "name": "get_coordinates_for_city"}
        
    ),
response = converse(messages, tools=[get_weather_schema, get_coordinates_for_city_schema, get_customer_schema, get_business_details_schema], tool_choice=tool_choice)
MAX_ITERATIONS = 5
iteration_count = 0
escalated = False

while (response.stop_reason == "tool_use") and (iteration_count < MAX_ITERATIONS):
    iteration_count += 1
    # Keep conversing until the model stops requesting tools
    add_assistant_message(messages, response.content)
    # response = converse(messages, tools=[get_customer_schema, get_business_details_schema])
    parsed_response = parse_response(response)
    # check if there is any tool use in the parsed response
    # Tool_response returns a list of result, one per tool actually requested, each tagged with its own tool_use_id
    tool_response = execute_tools(correlation_id, parsed_response, audit, user_query)
    print(f"tool_response length: {len(tool_response)}")
    for r in tool_response:
        print(r["tool_use_id"])
    # for each tool in the response create user message with the tool result and add it to the messages list
    user_message_content = []
    current_attempt = 0
    for result in tool_response:
        if result["result"].get("ok"):
            print("OK Tool Result: ", result)
            e = audit.create_event( "tool_result",
            correlation_id=correlation_id,
            tool_use_id=result["tool_use_id"],
            tool_name=result["result"].get("tool_name"),
            tool_input=json.dumps(result["result"]["tool_input"]),
            tool_output=json.dumps(result["result"]["data"]),
            model_version="claude-haiku-4-5"  )
            # print(e)
            # print(f"ABOUT TO LOG: {e['event_id']} iteration={iteration_count}")
            audit.log_event(e)
            user_message_content.append({"tool_use_id": result["tool_use_id"], "type": "tool_result", "content": [{"type": "text", "text": json.dumps(result["result"]["data"])}]})
        else:
            print(result)
            # Log Error Event with tool reported retryable, policy handling and classification conflict
            # If should retry then log the decision and branch on handling
            audit_event = audit.create_event(
                "error",
                correlation_id=correlation_id,
                error_category=result["result"].get("errorCategory"),
                error_type="tool_error",
                error_message=result["result"].get("message"),
                is_retryable=result["result"].get("toolReportedRetryable"),
                was_classified_as_default=result["result"].get("wasClassifiedAsDefault"),
                model_version="claude-haiku-4-5"
            )
            audit.log_event(audit_event)

            handling = retry_policy.config["error_classifications"].get(result["result"].get("errorCategory"), {}).get("handling")

            if handling == "escalate":
                # Escalate to human, log final answer event with escalated context
                # Log tool result with error message and tool use id
                escalated = True
                last_error_event_id = audit_event["event_id"]

                # Why does it need to go to model? 
                # user_message_content.append({"tool_use_id": result["tool_use_id"], "type": "tool_error", "content": [{"type": "text", "text": f"Tool {result['tool_use_id']} failed with error: {result['result'].get('message')}. Escalating to human as per policy."}]})

                final_answer_event = audit.create_event(
                    "final_answer",
                    correlation_id=correlation_id,
                    final_answer_context="escalated",
                    context_source_event_id=audit_event["event_id"],
                    final_answer_text="Escalated to human due to non-retryable error.",
                    model_version="claude-haiku-4-5"
                )
                audit.log_event(final_answer_event)

                break

            if handling == "return_to_model":
                # Feed error to model, loop continues
                user_message_content.append({"tool_use_id": result["tool_use_id"], "type": "tool_result", "is_error": True, "content": 
                                             [{"type": "text", "text": 
                                               json.dumps({
                                                    "errorCategory": result["result"].get("errorCategory"),
                                                    "message": result["result"].get("message"),
                                                    "attemptedInput": result["result"].get("attemptedInput")
                                               })}]
                })
                print("return to model.")
                continue

            if handling == "orchestrator_retry":

                if budget.exhausted():
                    should_retry, reason = False, f"global budget exhausted ({budget.used}/{budget.cap})"
                    #Log and Break out of loop and execute next tool. 

                # If there is no conflict with policy handling, then classify and decide
                classification, is_default = retry_policy.classify_error(result["result"].get("errorCategory"))
                print("CLASSIFICATION", classification)
                current_attempt = retry_count_by_category.get(result["result"].get("errorCategory"), 0)
                # Put it in the retry_count_by_category dict to keep track of the number of attempts for each error category
                retry_count_by_category[result["result"].get("errorCategory")] = current_attempt
                while current_attempt < classification["max_retries"]:

                    # As long as orchetrator_retry, audit the decision with any conflict with should_retry from tool
                    # Then call the tool again with the same inputs and timeout, and log the result
                    print("RESULT", result)
                    retry_decision_event = audit.create_event(
                        "retry_decision",
                        correlation_id=correlation_id,
                        error_source_event_id=audit_event["event_id"],
                        error_category=result["result"].get("errorCategory"),
                        decision="retry",
                        reason=result["result"]["errorCategory"] + ": attempt " + str(current_attempt) + " of " + str(classification["max_retries"]) + ", budget " + str(classification["max_retries"] - current_attempt) + " remaining",
                        was_classified_as_default=is_default,
                        retry_policy_version=retry_policy.policy_version,
                        retry_budget_remaining=classification["max_retries"] - current_attempt,
                        model_version="claude-haiku-4-5"
                    )
                    audit.log_event(retry_decision_event)
                    #Ececute the tool again with the same inputs and timeout
                    ms, escalate = compute_backoff(classification["backoff"], current_attempt, result["result"].get("retryAfter"))
                    if (escalate):
                        escalated = True
                        break
                    time.sleep(ms/1000)
                    retry_result = retry_tool_execution(audit_event=retry_decision_event, is_default=is_default, classification=classification, result=result)
                    current_attempt += 1
                    retry_count_by_category[result["result"].get("errorCategory")] = current_attempt
                    
                    if retry_result.get("ok"):
                        user_message_content.append({"tool_use_id": result["tool_use_id"], "type": "tool_result", "content": [{"type": "text", "text": json.dumps(retry_result["data"])}]})
                        break
                    else:
                        continue
                # append the user message after retry exhaustion or success
                escalated = True
                last_error_event_id = audit_event["event_id"]
                break
                # user_message_content.append({"tool_use_id": result["tool_use_id"], "type": "tool_result", "content": [{"type": "text", "text": f"Tool {result['tool_use_id']} failed with error: {result['result'].get('message')}. Retry attempts exhausted as per policy."}]})

            if handling is None:
                # If there is no handling defined in the policy, then escalate to human
                user_message_content.append({"tool_use_id": result["tool_use_id"], "type": "tool_result", "content": [{"type": "text", "text": f"Tool {result['tool_use_id']} failed with error: {result['result'].get('message')}. Escalating to human as per policy."}]})
                escalated = True
                final_answer_event = audit.create_event(
                    "final_answer",
                    correlation_id=correlation_id,
                    final_answer_context="escalated",
                    context_source_event_id=audit_event["event_id"],
                    final_answer_text="Escalated to human due to non-retryable error.",
                    model_version="claude-haiku-4-5"
                )
                audit.log_event(final_answer_event)
                break
    if escalated:
        last_error_event_id = audit_event["event_id"]
        response = "Tool " + result["result"]["tool_name"] + " failed after " + str(current_attempt)  + " retries due to " + result["result"]["errorCategory"] + ". Escalated for human review."
        break
    print(user_message_content)
    print(messages)
    add_user_message(messages, user_message_content)
    tool_choice = {"type": "auto"}
    response = converse(messages, tools=[get_business_details_schema, get_customer_schema, get_weather_schema], tool_choice=tool_choice)

# No Tool use or reached iteration count max
# At the end:
final_answer_event = audit.create_event(
    "final_answer",
    correlation_id=correlation_id,
    final_answer_context="normal" if not escalated else "escalated",
    context_source_event_id=last_error_event_id if escalated else None,
    final_answer_text=parse_final_response(response),

    model_version="claude-haiku-4-5"
)
audit.log_event(final_answer_event)

# Verify timeline:
timeline = audit.get_timeline(correlation_id)
for row in timeline:
    print(row)

PREREQS: []
tool_response length: 1
toolu_01QKyAY6JJWwFdt1ePsvhPsN
OK Tool Result:  {'tool_use_id': 'toolu_01QKyAY6JJWwFdt1ePsvhPsN', 'tool_name': 'get_coordinates_for_city', 'result': {'ok': True, 'data': {'latitude': 35.6895, 'longitude': 139.69171, 'city': 'Tokyo', 'country': 'Japan'}, 'tool_name': 'get_coordinates_for_city', 'tool_input': {'city': 'Tokyo'}}}
[{'tool_use_id': 'toolu_01QKyAY6JJWwFdt1ePsvhPsN', 'type': 'tool_result', 'content': [{'type': 'text', 'text': '{"latitude": 35.6895, "longitude": 139.69171, "city": "Tokyo", "country": "Japan"}'}]}]
[{'role': 'user', 'content': 'get weather for Tokyo'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_01QKyAY6JJWwFdt1ePsvhPsN', caller=DirectCaller(type='direct'), input={'city': 'Tokyo'}, name='get_coordinates_for_city', type='tool_use')]}]
PREREQS: [{'requires': 'get_coordinates_for_city', 'on': [('latitude', 'latitude'), ('longitude', 'longitude')], 'condition': None, 'allow_user_supplied': True, 'match_fields': ['lat